> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAG3LntaIPg/NeNJCP8AM5XBnxItQhTasA/view?utm_content=DAG3LntaIPg&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h7d4bc9e2d1)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install "langgraph-cli[inmem]" openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 实践代码

为了能够顺利的演示内置中间件的使用详情，这里我们使用一段简单的智能体代码演示：

In [ ]:
from langchain_community.chat_models import ChatTongyi
import os
llm = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")

from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["arxiv"])

from langgraph.checkpoint.memory import InMemorySaver 
memory = InMemorySaver()

from langchain.agents import create_agent
agent = create_agent(model=llm, 
                     tools=tools, 
                     system_prompt="You are a helpful assistant", 
                     checkpointer=memory)

# 2. 内置中间件

## 2.1 简介

针对于一些智能体开发常用的中间件，LangChain 官方团队给出了一系列的内置中间件来帮助大家进行快速上手。这些中间件包括：
- ModelFallbackMiddleware：主模型失败时自动换模型
- ModelCallLimitMiddleware：限制模型调用次数
- ToolCallLimitMiddleware：限制工具调用频率
- ToolRetryMiddleware：工具失败时自动重试
- LLMToolEmulator：用LLM模拟工具执行
- SummarizationMiddleware：摘要对话历史
- ContextEditingMiddleware：清理旧上下文
- PIIMiddleware：检测/脱敏个人隐私信息
- HumanInTheLoopMiddleware：人工审批确认机制
- ...

## 2.2 ModelFallbackMiddleware：模型容灾与降级策略

ModelFallbackMiddleware 是在主模型失败时自动切换备用模型，保持任务不中断。

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

agent = create_agent(
  model=ChatTongyi(model="xxx"), # 错误模型
  tools=tools,
  system_prompt="You are a helpful assistant",
  middleware=[ModelFallbackMiddleware(
      ChatTongyi(model="qwen-plus"),
      ChatTongyi(model="qwen-turbo",
      checkpointer=memory))])

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386"}]}, config={"configurable": {"thread_id": "user_1"}})

print(result1["messages"][-1].content)

## 2.3 ModelCallLimitMiddleware：限制模型调用次数
ModelCallLimitMiddleware 的作用就是限定模型调用次数，一旦达到上限，就让 Agent 自动停止或报错。

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

middleware_model_limit = ModelCallLimitMiddleware(thread_limit=2, run_limit=1)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant",
    middleware=[middleware_model_limit],
    checkpointer=InMemorySaver()
)

config3 = {"configurable": {"thread_id": "user_3"}}

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386，100字即可"}]}, config=config3)
result2 = agent.invoke({"messages": [{"role": "user", "content": "你能总结一下吗？100字即可"}]}, config=config3)
result3 = agent.invoke({"messages": [{"role": "user", "content": "推荐几篇拓展阅读吧"}]}, config=config3)

print(result3["messages"][-1].content)

## 2.4 ToolCallLimitMiddleware：限制工具调用次数

ToolCallLimitMiddleware ，通过在调用工具的前后设置调用次数上限，防止 Agent 过度调用工具或陷入无穷循环。

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

middleware=[
    ToolCallLimitMiddleware(run_limit=10),  # 全局
    ToolCallLimitMiddleware(tool_name="arxiv", thread_limit=1),  # 局部
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant",
    middleware=middleware,
    checkpointer=InMemorySaver()
)

config3 = {"configurable": {"thread_id": "user_3"}}

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386，100字即可"}]}, config=config3)
result2 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 2103.08386，100字即可"}]}, config=config3)


print(result2["messages"][-2].content)


## 2.5 ToolRetryMiddleware：工具调用自动重试机制

ToolRetryMiddleware 不仅仅可以在调用工具的时候重复多次，还能够设置延时、指数回退（exponential backoff）、随机扰动（jitter）等机制。从而让智能体在调用不稳定外部服务（例如网络请求、搜索接口、数据库API）时更加鲁棒（robust）。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware

agent = create_agent(
  model=llm,
  tools=tools,
  system_prompt="You are a helpful assistant",
  middleware=[
    ToolRetryMiddleware(
      max_retries=3,   # 最多重试 3 次
      backoff_factor=2.0, # 指数退避倍数
      initial_delay=1.0, # 首次延迟 1 秒
      max_delay=60.0,   # 最大等待时间 60 秒
      jitter=True,    # 添加随机扰动，防止雪崩重试
    ),
  ],
  checkpointer=memory)

## 2.6 LLMToolEmulator：用模型“假装”执行工具
LLMToolEmulator可以通过用 LLM 来模拟工具的执行结果（tool emulation）。这样就可以让整体流程先顺利进行，后面再补充相对应的逻辑。


In [ ]:
from langchain.agents.middleware import LLMToolEmulator

agent = create_agent(
    model=llm,
    tools=tools,
    middleware=[
        LLMToolEmulator(
            model = ChatTongyi(model="qwen-turbo"),
            tools=["arxiv"],
        ), 
    ],
)

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386"}]}, config={"configurable": {"thread_id": "user_1"}})
print(result1["messages"][-1].content)

当我们希望所有的工具都用 LLM 进行模拟，我们可以这样写入（tools 参数里传入 None）：

In [ ]:
from langchain.agents.middleware import LLMToolEmulator
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    ''' 获取指定位置的天气信息 '''
    message = f"获取 {location} 的天气信息"
    print(message)  
    return message

@tool
def search_database(query: str) -> str:
    ''' 从数据库中搜索相关信息 '''
    message = f"数据库中关于 {query} 的信息是：..."
    print(message)  
    return message  
    
@tool
def send_email(to: str, subject: str, body: str) -> str:
    ''' 发送邮件 '''
    message = f"已发送邮件至 {to}，主题为 {subject}，内容为 {body}"
    print(message)  
    return message

agent = create_agent(
    model=llm,
    tools=[get_weather, search_database, send_email],
    middleware=[
        LLMToolEmulator(
            model = ChatTongyi(model="qwen-turbo")
        ),  # 所有工具调用都由 LLM 模拟执行
    ],
)

# 调用时，模型不会真的去查数据库或发邮件
result = agent.invoke({"messages": [{"role": "user", "content": "查一下广州的天气"}]})

print(result["messages"][-1].content)


## 2.7 LLMToolSelectorMiddleware：智能工具筛选器

LLMToolSelectorMiddleware 的作用就是在主模型调用前，让一个小模型（或更快的模型）先分析用户意图 → 从众多工具中选出相关的几个。

In [ ]:
from langchain.agents.middleware import LLMToolSelectorMiddleware

agent = create_agent(
    model=llm,
    tools=[get_weather, search_database, send_email],
    middleware=[
        LLMToolSelectorMiddleware(
            model=ChatTongyi(model="qwen-turbo"),  # ✅ 选择辅助模型用于筛选
            max_tools=2,                 # 最多保留 2 个
            always_include=["send_email"],   # 某些关键工具始终保留
        ),
    ],
)

result = agent.invoke({"messages": [{"role": "user", "content": "查一下广州的天气"}]})

print(result["messages"][-1].content)

## 2.8 TodoListMiddleware：任务拆解与规划能力
在复杂任务中，模型常出现：
- 一步把所有事情混在一起做
- 任务顺序混乱
- 忘记步骤
- 无法跟踪进度
- 无法形成一个持久化的任务列表

这类问题在大型项目（写代码、调试、多文件编辑）中尤为明显。

因此 LangChain 提供 TodoListMiddleware，让智能体内部自动拥有“规划→执行→更新”的能力。

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

agent = create_agent(
    model=llm,
    tools=tools,
    middleware=[TodoListMiddleware()],
)

task = '''请帮我完成以下任务：

1. 使用 arxiv 搜索论文 1605.08386，并提取主要贡献。
2. 搜索论文 1605.08387，并比较两篇论文的研究主题差异。
3. 最后写一个 300 字总结，说明它们在研究方法上的共同点。

请分步骤执行，并进行规划。'''

result1 = agent.invoke({"messages": [{"role": "user", "content": f"{task}"}]}, config={"configurable": {"thread_id": "user_1"}})
print(result1)

我们可以通过将智能体部署在 LangSmith Studio 上查看到更清晰的调用结果。详见 todo_list_agent 文件夹并进行部署。